# Lobbia LP Analysis — Usage Examples

This notebook demonstrates how to use the `lobbia_lp` package to analyze Langmuir probe I-V traces following Lobbia & Beal (2017) recommended practices.

## Setup

Import libraries and configure plotting:

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from lobbia_lp import analyze, load_trace, io, potentials, electron as electron_module

# Set up plotting style
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (14, 10)

## QUICK START: Batch Process Your Axial Langmuir Directory

**Just run these three cells in order:**

### Step 1: Load data and configure analysis parameters

In [2]:
# Your Axial Langmuir data directory
data_dir = Path("/Users/braden/Documents/ProbeTools/Claude_LP_Sandbox/Axial Langmuir")

# Probe parameters (customize if needed)
probe_params = {
    'probe_radius_m': 50.8e-6,          # Probe radius [m]
    'probe_length_m': 2.54e-3,          # Probe length [m]
    'probe_area_m2': np.pi * 2 * 50.8e-6 * (2.54e-3 + 50.8e-6),  # Cylindrical surface area
    'gas': 'kr',                         # Krypton (change to 'xe', 'ar', etc. if needed)
    'probe_geometry': 'cylindrical',     # cylindrical, spherical, or planar
}

# Find all CSV files
trace_files = sorted(data_dir.glob("*.csv"))
print(f"✓ Found {len(trace_files)} CSV files in {data_dir.name}")
print(f"  Gas: {probe_params['gas'].upper()}")
print()

✓ Found 28 CSV files in Axial Langmuir
  Gas: KR



### Step 2: Analyze all traces

In [3]:
# Process all traces
results_list = []

print(f"Analyzing {len(trace_files)} traces...\n")

for i, trace_file in enumerate(trace_files, 1):
    try:
        V, I = load_trace(str(trace_file))
        result = analyze(
            V, I,
            **probe_params,
            max_iterations=10,
            verbose=False
        )
        
        results_list.append({
            'File': trace_file.name,
            'Vf (V)': result.vf,
            'Vp (V)': result.vp,
            'Te (eV)': result.te,
            'ne (m⁻³)': result.ne,
            'ni (m⁻³)': result.ni,
            'n (m⁻³)': result.n,
            'λD (m)': result.debye_length,
            'rp/λD': result.rp_lambda,
            'Sheath': result.sheath_regime,
            'Converged': result.converged,
        })
        
        print(f"[{i:2d}/{len(trace_files)}] ✓ {trace_file.name:30s} Te={result.te:6.2f} eV, ne={result.ne:.2e} m⁻³")
        
    except Exception as e:
        print(f"[{i:2d}/{len(trace_files)}] ✗ {trace_file.name:30s} ERROR: {e}")

print(f"\n✓ Successfully analyzed {len(results_list)} traces")

Analyzing 28 traces...

[ 1/28] ✓ Kr_15sccm_10A.csv              Te=  3.68 eV, ne=3.03e+16 m⁻³
[ 2/28] ✓ Kr_15sccm_10A2.csv             Te=  3.69 eV, ne=3.69e+16 m⁻³
[ 3/28] ✓ Kr_15sccm_11A.csv              Te=  1.74 eV, ne=6.08e+16 m⁻³
[ 4/28] ✓ Kr_15sccm_12A.csv              Te=  1.75 eV, ne=6.92e+16 m⁻³
[ 5/28] ✓ Kr_15sccm_12p5A.csv            Te=  1.90 eV, ne=6.10e+16 m⁻³
[ 6/28] ✓ Kr_15sccm_13A.csv              Te=  1.80 eV, ne=7.45e+16 m⁻³
[ 7/28] ✓ Kr_15sccm_14A.csv              Te=  1.84 eV, ne=7.78e+16 m⁻³
[ 8/28] ✓ Kr_15sccm_15A.csv              Te=  1.74 eV, ne=7.22e+16 m⁻³
[ 9/28] ✓ Kr_15sccm_16A.csv              Te=  1.91 eV, ne=6.78e+16 m⁻³
[10/28] ✓ Kr_15sccm_17A.csv              Te=  1.98 eV, ne=6.35e+16 m⁻³
[11/28] ✓ Kr_15sccm_17p5A.csv            Te=  1.94 eV, ne=6.26e+16 m⁻³
[12/28] ✓ Kr_15sccm_18A.csv              Te=  2.05 eV, ne=5.83e+16 m⁻³
[13/28] ✓ Kr_15sccm_19A.csv              Te=  2.18 eV, ne=6.19e+16 m⁻³
[14/28] ✓ Kr_15sccm_20A.csv              Te=  2.15 eV

### Step 3: Save plots for each trace (THIS CREATES THE plots/ DIRECTORY)

In [4]:
# Create output directory for plots
plot_dir = data_dir / "plots"
plot_dir.mkdir(parents=True, exist_ok=True)

print(f"Creating individual plots in: {plot_dir}\n")

# Define plot functions
def save_iv_plot(V, I, result, output_path):
    """Save I-V characteristic plot with Vf and Vp marked."""
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.plot(V, I*1e3, 'k-', linewidth=2, label='Measured trace')
    ax.axvline(result.vf, color='red', linestyle='--', linewidth=2, alpha=0.7, label=f'Vf = {result.vf:.2f} V')
    ax.plot(result.vf, np.interp(result.vf, V, I)*1e3, 'ro', markersize=10, markerfacecolor='red', markeredgewidth=2, markeredgecolor='darkred')
    ax.axvline(result.vp, color='blue', linestyle='--', linewidth=2, alpha=0.7, label=f'Vp = {result.vp:.2f} V')
    ax.plot(result.vp, np.interp(result.vp, V, I)*1e3, 'bo', markersize=10, markerfacecolor='blue', markeredgewidth=2, markeredgecolor='darkblue')
    ax.grid(True, alpha=0.3)
    ax.set_xlabel('Bias Voltage [V]', fontsize=12)
    ax.set_ylabel('Probe Current [mA]', fontsize=12)
    ax.set_title('I-V Characteristic', fontsize=13, fontweight='bold')
    ax.legend(fontsize=11, loc='best')
    plt.tight_layout()
    plt.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.close()

def save_semilog_plot(V, I, result, output_path):
    """Save semilog fit plot for Te extraction."""
    try:
        V_norm, I_norm = io._normalize_trace(V, I)
        vf, _ = potentials.find_floating_potential(V_norm, I_norm)
        vp, _, _ = potentials.find_plasma_potential(V_norm, I_norm, vf)
        
        ion_mask = V_norm <= vf
        Ii_prelim = np.polyfit(V_norm[ion_mask], I_norm[ion_mask], 1)
        Ii_prelim = np.poly1d(Ii_prelim)(V_norm)
        Ii_prelim = np.minimum(Ii_prelim, 0.0)
        
        Ie = I_norm - Ii_prelim
        Ie = np.maximum(Ie, 1e-15)
        te, d_te, fit_info = electron_module.fit_electron_temperature(V_norm, Ie, vf, vp)
        
        fit_mask = fit_info['fit_window_mask']
        V_fit = V_norm[fit_mask]
        Ie_fit = Ie[fit_mask]
        slope = fit_info['slope']
        intercept = fit_info['intercept']
        ln_Ie_fit_line_full = slope * V_fit + intercept
        Ie_fit_line_full = np.exp(ln_Ie_fit_line_full)
        
        fig, ax = plt.subplots(figsize=(12, 7))
        electron_region = V_norm >= vf
        ax.semilogy(V_norm[electron_region], Ie[electron_region], 'k.', markersize=6, alpha=0.5, label='Ie (all data)')
        ax.semilogy(V_fit, Ie_fit, 'b.-', markersize=8, linewidth=2, alpha=0.8, label='Ie (fit region)')
        ax.semilogy(V_fit, Ie_fit_line_full, 'r-', linewidth=3, label=f'Linear fit (Te = {te:.2f} ± {d_te:.2f} eV)', zorder=10)
        ax.axvline(vf, color='red', linestyle='--', alpha=0.5, linewidth=1.5)
        ax.axvline(vp, color='blue', linestyle='--', alpha=0.5, linewidth=1.5)
        ax.text(vf, ax.get_ylim()[1]*0.5, f'Vf={vf:.2f}V', rotation=90, va='top', ha='right', fontsize=10, color='red')
        ax.text(vp, ax.get_ylim()[1]*0.5, f'Vp={vp:.2f}V', rotation=90, va='top', ha='right', fontsize=10, color='blue')
        ax.grid(True, alpha=0.3, which='both')
        ax.set_xlabel('Bias Voltage [V]', fontsize=12)
        ax.set_ylabel('Electron Current [A] (log scale)', fontsize=12)
        ax.set_title('Semilog Fit for Te', fontsize=13, fontweight='bold')
        ax.legend(fontsize=11, loc='best')
        ax.set_xlim([vf - 0.5, vp + 1])
        plt.tight_layout()
        plt.savefig(output_path, dpi=150, bbox_inches='tight')
        plt.close()
    except Exception as e:
        pass  # Silently skip semilog if there's an error

# Generate plots for all files
plot_count = 0
for i, trace_file in enumerate(trace_files, 1):
    try:
        V, I = load_trace(str(trace_file))
        result = analyze(V, I, **probe_params, max_iterations=10, verbose=False)
        
        stem = trace_file.stem
        iv_path = plot_dir / f"{stem}_IV_characteristic.png"
        semilog_path = plot_dir / f"{stem}_Te_semilog_fit.png"
        
        save_iv_plot(V, I, result, iv_path)
        save_semilog_plot(V, I, result, semilog_path)
        plot_count += 2
        
        print(f"[{i:2d}/{len(trace_files)}] ✓ {trace_file.name:30s} → 2 plots saved")
    except Exception as e:
        print(f"[{i:2d}/{len(trace_files)}] ✗ {trace_file.name:30s} → ERROR: {e}")

print(f"\n✅ Successfully saved {plot_count} plots!")
print(f"📁 Location: {plot_dir}/")

Creating individual plots in: /Users/braden/Documents/ProbeTools/Claude_LP_Sandbox/Axial Langmuir/plots

[ 1/28] ✓ Kr_15sccm_10A.csv              → 2 plots saved
[ 2/28] ✓ Kr_15sccm_10A2.csv             → 2 plots saved
[ 3/28] ✓ Kr_15sccm_11A.csv              → 2 plots saved
[ 4/28] ✓ Kr_15sccm_12A.csv              → 2 plots saved
[ 5/28] ✓ Kr_15sccm_12p5A.csv            → 2 plots saved
[ 6/28] ✓ Kr_15sccm_13A.csv              → 2 plots saved
[ 7/28] ✓ Kr_15sccm_14A.csv              → 2 plots saved
[ 8/28] ✓ Kr_15sccm_15A.csv              → 2 plots saved
[ 9/28] ✓ Kr_15sccm_16A.csv              → 2 plots saved
[10/28] ✓ Kr_15sccm_17A.csv              → 2 plots saved
[11/28] ✓ Kr_15sccm_17p5A.csv            → 2 plots saved
[12/28] ✓ Kr_15sccm_18A.csv              → 2 plots saved
[13/28] ✓ Kr_15sccm_19A.csv              → 2 plots saved
[14/28] ✓ Kr_15sccm_20A.csv              → 2 plots saved
[15/28] ✓ Kr_15sccm_21A.csv              → 2 plots saved
[16/28] ✓ Kr_15sccm_22A.csv             

## Results Summary

In [5]:
# Create and display results table
df_summary = pd.DataFrame(results_list)

# Save to CSV
csv_file = data_dir / "analysis_summary.csv"
df_summary.to_csv(csv_file, index=False)

print(f"Summary of {len(df_summary)} analyses:\n")
pd.options.display.float_format = '{:.3e}'.format
print(df_summary.to_string(index=False))
print(f"\n✓ CSV saved to: {csv_file}")

Summary of 27 analyses:

                File    Vf (V)    Vp (V)   Te (eV)  ne (m⁻³)  ni (m⁻³)   n (m⁻³)    λD (m)     rp/λD Sheath  Converged
   Kr_15sccm_10A.csv 9.871e+00 1.991e+01 3.678e+00 3.032e+16 1.182e+18 6.061e+17 8.189e-05 6.204e-01    oml       True
  Kr_15sccm_10A2.csv 1.053e+01 2.078e+01 3.690e+00 3.688e+16 1.350e+18 6.937e+17 7.436e-05 6.832e-01    oml       True
   Kr_15sccm_11A.csv 1.131e+01 1.771e+01 1.742e+00 6.076e+16 1.095e+18 5.779e+17 3.980e-05 1.276e+00    oml       True
   Kr_15sccm_12A.csv 9.760e+00 1.660e+01 1.750e+00 6.916e+16 7.043e+17 3.867e+17 3.740e-05 1.358e+00    oml       True
 Kr_15sccm_12p5A.csv 9.589e+00 1.676e+01 1.895e+00 6.103e+16 6.759e+17 3.685e+17 4.143e-05 1.226e+00    oml       True
   Kr_15sccm_13A.csv 9.875e+00 1.712e+01 1.800e+00 7.453e+16 6.964e+17 3.854e+17 3.654e-05 1.390e+00    oml       True
   Kr_15sccm_14A.csv 9.833e+00 1.733e+01 1.844e+00 7.779e+16 7.053e+17 3.915e+17 3.620e-05 1.403e+00    oml       True
   Kr_15sccm_15A.csv 9.

---

## Additional Examples (Optional)

### Analyze a single file with detailed output:

In [6]:
# Example: Analyze a single file
single_file = data_dir / trace_files[0].name  # First file

V, I = load_trace(str(single_file))
result = analyze(V, I, **probe_params, verbose=True)

print("\n" + "="*60)
print("ANALYSIS RESULTS")
print("="*60)
df_results = result.to_dataframe()
print(df_results.to_string())
print(f"\nSheath regime: {result.sheath_regime}")
print(f"Iterations: {result.n_iterations}")
print(f"Converged: {result.converged}")

Step 2: Vf = 9.871 V ± 0.166 V
Step 5: Vp = 19.907 V ± 0.500 V
Step 6: Te = 3.678 eV ± 0.137 eV
Step 7: ne = 3.032e+16 m^-3 ± 2.144e+15 m^-3
  Iteration 0: ni = 1.182e+18 m^-3, rp/λD = 0.6, regime = oml
  Iteration 1: ni = 1.182e+18 m^-3, rp/λD = 0.6, regime = oml
Converged after 2 iterations

ANALYSIS RESULTS
               Value  Uncertainty  Unit
Vf         9.871e+00    3.812e-01     V
Vp         1.991e+01    1.414e-01     V
Te         3.678e+00    5.523e-02    eV
n          6.061e+17    2.955e+17  m^-3
ne         3.032e+16    1.728e+15  m^-3
ni         1.182e+18    5.910e+17  m^-3
Ie_sat     1.289e-03    1.289e-05     A
Ii_sat    -4.250e-04    4.250e-05     A
lambda_D   8.189e-05          NaN     m
rp_lambda  6.204e-01          NaN     1

Sheath regime: oml
Iterations: 2
Converged: True
